# Cross-Model Verification

Load the per-model semantic-match results produced by
`rag-sym-semantic-match-vllm.ipynb` (one JSONL per model) and run a
majority vote per generated symptom. The vote count is then used as
a proxy for cross-model agreement / hallucination rate.

**Inputs**: 7 model output files, all aligned row-by-row.
**Output**: one `majority_results` list with the consensus label,
vote count, and full per-model voting record for each row.


## 1. Load all per-model outputs and run the majority vote

Each file has the same number of rows in the same order — row `i`
is the *same* generated symptom across all models. For each row we
tally `onto_best_match` strings and pick whichever label collected
the most votes.


In [ ]:
import json

# All paths relative to the notebook location (the `code/` folder).
files = [
    "../outs/semsymp-analysis-gemma-3-4b-it.jsonl",
    "../outs/semsymp-analysis-gemma-7b-it.jsonl",
    "../outs/semsymp-analysis-Llama-3.1-8B.jsonl",
    "../outs/semsymp-analysis-mistral-02102026.jsonl",
    "../outs/semsymp-analysis-phi-02102026.jsonl",
    "../outs/semsymp-analysis-qwen-02102026.jsonl",
    "../outs/semsymp-analysis-Qwen2.5-14B-Instruct.jsonl",
]

# all_models[i] is the full list of records produced by model i.
all_models = [[json.loads(line) for line in open(f)] for f in files]

majority_results = []

for i in range(len(all_models[0])):
    # One vote per model: each model's best_match for row i.
    matches = [m[i]["onto_best_match"] for m in all_models]

    # Tally votes (None counts as a distinct vote so models that
    # abstained don't get silently re-aligned to a real label).
    vote_counts = {}
    for m in matches:
        vote_counts[m] = vote_counts.get(m, 0) + 1

    # Pick the label with the highest vote count.
    best_term = max(vote_counts, key=vote_counts.get)
    votes = vote_counts[best_term]

    majority_results.append({
        "generated_symp": all_models[0][i]["generated_symptom"],
        "majority_term": best_term,
        "votes": votes,
        "all_model_matches": matches,
        "status": "verified" if votes >= 3 else "no-consensus",
    })


## 2. Bucket each row by vote strength

With 7 models voting:
- `>= 4` votes → **majority** (strict majority of voters agree).
- `2 or 3` votes → **minority / plurality** (some agreement but no
  strict majority).
- `1` vote → **hallucination** (every model produced a different
  label — likely fabrication).


In [ ]:
majority_counter = 0
minority_counter = 0
no_match_counter = 0

for result_rec in majority_results:
    if result_rec["votes"] >= 4:
        majority_counter += 1
    elif 2 <= result_rec["votes"] < 4:
        minority_counter += 1
    else:
        # Bug fix: was `no_match_counter = 0` which clobbered the
        # counter on every iteration. Must be an increment.
        no_match_counter += 1


## 3. Report the per-bucket breakdown

In [33]:
total = len(majority_results)

print("Majority Counter: ", majority_counter)
print("Minority Counter: ", minority_counter)
print("No Match Counter: ", no_match_counter)
print("---------------------------------------")
print("Majority %: ", majority_counter / total * 100)
print("Minority %:", round(minority_counter / total * 100, 2))
print("Hallucination %: ", no_match_counter / total * 100)
print("---------------------------------------")


Majority Counter:  7685
Minority Counter:  2315
No Match Counter:  0
---------------------------------------
Majority %:  76.85
Minority %: 23.15
Hallicination %:  0.0
---------------------------------------


## 4. Sanity check — buckets should sum to N rows

In [26]:
majority_counter + minority_counter + no_match_counter


10000